# Milestone 5 Lab, notebook 3 of 7: AI reads a changed file

## Where we are

| Notebook | What you do |
|---|---|
| 1. Files arrive, by hand | Load the files the naive way, and watch three things go wrong |
| 2. A loader that is safe to repeat | Fingerprint each file, check its columns, and choose a rule for every file |
| **3. AI reads a changed file** (you are here) | Ask a model what a renamed column means, and score its suggestions |
| 4. A workflow that can wait | Build the loader as a graph that pauses for your approval |
| 5. Crash and carry on | Restart the kernel on purpose, and find the file still waiting |
| 6. A swarm of file workers | Load fifty files side by side, and score the run for quality, cost, and time |
| 7. The same question, from loaded files | Approve the waiting files, run the pipeline, and ask the leader's question |

Model requests in this notebook: **about 9**. A full run of this lab costs about 3 US cents.

## What this notebook is for

Six files have columns the loader does not expect. Code found them. Code cannot say what `list_price` or `cost_1` **mean**.

A data engineer would open the file, look at the names and a few values, compare them with an older file, and decide. That is reading and judgement, which is what a language model is good at.

In this notebook a model suggests a **mapping** for each file, and you score it against the truth. You also see why a person must still approve it.

> This notebook needs `OPENROUTER_API_KEY` in your `.env` file.

In [ ]:
import pandas as pd
from lab_helpers import LAB, LANDING, connect, reset_lab, show_doc

reset_lab()

## Step 1: Who does what

In [ ]:
show_doc("HLD.md", "6. Where AI is used and where it is not")

## Step 2: Load the files that need no decision

The model will want an older file to compare with. So first we load everything that loads by rule. `load_batch` is the production loader. You build its parts in notebooks 4 to 6. For now, read its result.

In [ ]:
from lab_helpers import count_outcomes

from retail_profit_stack.ingest.dispatch import load_batch

results = load_batch(places=LAB)

count_outcomes(results)

42 files loaded, one was skipped, one was rejected, and six are **waiting**. Nothing from those six has been loaded.

In [ ]:
waiting = [item["file_name"] for item in results if item["outcome"] == "waiting"]
waiting

## Step 3: The simple approach. Names only

Start with the least we could give a model: the column names, and nothing else.

In [ ]:
client, MODEL = connect()

In [ ]:
def ask(prompt):
    """Send one prompt to the model and return its reply as text."""
    reply = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=1500
    )
    return reply.choices[0].message.content


names_only = """A shipments file should have the columns pick_pack_cost and carrier_cost.
The file that arrived has cost_1 and cost_2 in their place.

Which is which? Answer in two sentences."""

print(ask(names_only))

Read the reply. The names `cost_1` and `cost_2` say nothing, so the model can only guess from their order, or say it cannot tell. Either way, nobody should load a file on that.

If the cell failed with a network error, run it again.

## Step 4: Give the model what a person would look at

A person would look at the values, and at an older file. `mapping_prompt` puts those in the prompt: the expected columns and their meanings, five rows of the new file, and five rows of an earlier file from the same feed.

In [ ]:
from retail_profit_stack.ingest.feeds import expected_columns
from retail_profit_stack.ingest.mapping import earlier_rows, mapping_prompt
from retail_profit_stack.ingest.rules import inspect_file

file = LANDING / "shipments_2026-10-26.csv"
found = inspect_file(file, expected_columns("shipments"), loaded={}, approved=[])

prompt = mapping_prompt("shipments", file, found.missing, found.unknown, earlier_rows(LAB, "shipments"))
print(prompt)

Look at the two samples. In the earlier file, `pick_pack_cost` is 2.50 and `carrier_cost` is 4.00 for the same product. In the new file, `cost_1` is 2.50 and `cost_2` is 4.00. The values give the answer the names did not.

The instructions are short. Read the last sentence.

In [ ]:
from retail_profit_stack.ingest.mapping import MAPPING_INSTRUCTIONS

print(MAPPING_INSTRUCTIONS)

## Step 5: A structured suggestion

We do not want a paragraph back. We want an answer code can read: one entry per unknown column, a confidence, and a note. `suggest_mapping` asks for exactly that shape, the way milestone 1 asked for a structured answer.

In [ ]:
from retail_profit_stack.agent import make_model
from retail_profit_stack.ingest.mapping import suggest_mapping

model = make_model(timeout=30)

suggestion, cost = suggest_mapping(model, "shipments", file, found.missing, found.unknown, LAB)

pd.DataFrame([guess.model_dump() for guess in suggestion.columns])

In [ ]:
print("Confidence:", suggestion.confidence)
print("Note:", suggestion.note)
print("As a mapping:", suggestion.as_mapping())
print("This call cost", round(cost, 5), "US dollars")

Notice three things. `warehouse_code` is mapped to "none": it is an extra column, and the model said so. The **confidence** is likely "low", because the model was told to say so when names give no clue. And the note tells the person what to check.

## Step 6: All six files, scored

We planted these changes, so we know the true mapping of each file.

In [ ]:
from retail_profit_stack.ingest.feeds import feed_of

suggested = {}
confidence = {}
total_cost = 0.0

for file_name in waiting:
    feed = feed_of(file_name)
    file = LANDING / file_name
    found = inspect_file(file, expected_columns(feed), loaded={}, approved=[])
    suggestion, cost = suggest_mapping(model, feed, file, found.missing, found.unknown, LAB)
    suggested[file_name] = suggestion.as_mapping()
    confidence[file_name] = suggestion.confidence
    total_cost += cost
    print(file_name, suggestion.confidence, suggestion.as_mapping())

In [ ]:
from retail_profit_stack.ingest.mapping import score_suggestions

scores = score_suggestions(suggested)
scores["confidence"] = scores["file_name"].map(confidence)

print("All six cost", round(total_cost, 4), "US dollars")
scores[["file_name", "suggested", "result", "confidence"]]

When the lab was built, the model got all six right in every run. It always marked the `cost_1` file as low confidence, and sometimes one other file too. Yours should be the same or close. If a suggestion is wrong, that is the lesson of the next step.

## Step 7: Code applies a mapping

A mapping is applied by code, never by the model. `read_file` renames the columns, drops the extra ones, and checks every value against the contract.

In [ ]:
from retail_profit_stack.ingest.mapping import read_file

right = {"cost_1": "pick_pack_cost", "cost_2": "carrier_cost"}
table = read_file(LANDING / "shipments_2026-10-26.csv", "shipments", right)

table.to_pandas().head(3)

## Step 8: Why a person must still approve

Now apply the **wrong** mapping, with the two columns swapped.

In [ ]:
swapped = {"cost_1": "carrier_cost", "cost_2": "pick_pack_cost"}
table = read_file(LANDING / "shipments_2026-10-26.csv", "shipments", swapped)

table.to_pandas().head(3)

No error. Both columns hold amounts, so the swapped file passes every check code can make. It would load, and a week of fulfilment and shipping costs would quietly trade places.

Code catches some bad mappings. Here is one it does catch: text where an amount belongs.

In [ ]:
from retail_profit_stack.transform.contracts import ContractError

nonsense = {"cost_1": "pick_pack_cost", "warehouse_code": "carrier_cost"}

try:
    read_file(LANDING / "shipments_2026-10-26.csv", "shipments", nonsense)
except (ContractError, Exception) as error:
    print("Refused:", str(error)[:160])

So there are three lines of defence, and none is enough alone:

- The **model** reads the values and marks what it is unsure of.
- **Code** refuses a mapping whose values do not fit.
- A **person** sees the suggestion, its reasons, and the sample rows, and decides.

A mapping that swaps two amounts gets past code. Only the model's caution and the person can stop it. That is why the business rule says a person approves **every** mapping.

## What we now know

- With names only, a model can only guess what `cost_1` means. With sample values and an earlier file, it can work it out.
- The suggestion is **structured**: a mapping, a reason for each column, a confidence, and a note.
- The model's suggestions were scored against the truth we planted.
- Code applies a mapping and checks the values. A swap of two amounts passes that check.
- So a person approves every mapping. The model **suggests**; it never decides.

**Next:** notebook 4. If a person must approve, the loader must be able to **wait** for them.